In [1]:
import sys
import importlib.util

print("Python:", sys.version.split()[0])

packages = [
    "ollama",
    "llama_index",
    "langchain",
    "langgraph",
    "haystack",
    "chromadb",
]

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if installed else 'Not installed'}")

Python: 3.14.7
ollama: Installed
llama_index: Installed
langchain: Installed
langgraph: Installed
haystack: Installed
chromadb: Installed


In [1]:
import subprocess

try:
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True,
        text=True,
        timeout=5,
    )
    print(result.stdout or result.stderr)
except Exception as e:
    print(type(e).__name__, "-", e)

NAME                       ID              SIZE      MODIFIED    
nomic-embed-text:latest    0a109f422b47    274 MB    10 days ago    
qwen2.5:3b                 357c53fb659c    1.9 GB    3 weeks ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    3 weeks ago    



In [3]:
import urllib.request

try:
    with urllib.request.urlopen(
        "http://localhost:11434/api/tags", timeout=3
    ) as response:
        print("Ollama API status:", response.status)
        print(response.read().decode()[:500])
except Exception as error:
    print(type(error).__name__, "-", error)

Ollama API status: 200
{"models":[{"name":"nomic-embed-text:latest","model":"nomic-embed-text:latest","modified_at":"2026-10-07T22:14:21.0791574+05:30","size":274302450,"digest":"0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f","details":{"parent_model":"","format":"gguf","family":"nomic-bert","families":["nomic-bert"],"parameter_size":"137M","quantization_level":"F16","context_length":2048,"embedding_length":768,"runner":"ggml"},"capabilities":["embedding"]},{"name":"qwen2.5:3b","model":"qwen2.5:3b",


In [4]:
import json
import urllib.request

payload = json.dumps({
    "model": "nomic-embed-text:latest",
    "input": "Local AI research assistant document search"
}).encode("utf-8")

request = urllib.request.Request(
    "http://localhost:11434/api/embed",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(request, timeout=10) as response:
        data = json.loads(response.read().decode("utf-8"))
    print("Embedding generated:", bool(data.get("embeddings")))
    print("Embedding dimensions:", len(data["embeddings"][0]))
except Exception as error:
    print(type(error).__name__, "-", error)

Embedding generated: True
Embedding dimensions: 768


In [5]:
# Small local knowledge base for our first retrieval test
documents = [
    {
        "title": "RAG Basics",
        "text": "Retrieval-Augmented Generation retrieves relevant information from documents and uses it to help generate grounded answers."
    },
    {
        "title": "Embeddings",
        "text": "Embeddings represent text as numerical vectors. Similarity between vectors can help identify relevant document chunks."
    },
    {
        "title": "Local AI",
        "text": "Ollama can run language models and embedding models locally through its API."
    },
]

print("Documents loaded:", len(documents))
for document in documents:
    print("-", document["title"])

Documents loaded: 3
- RAG Basics
- Embeddings
- Local AI


In [6]:
import json
import urllib.request

def get_embedding(text):
    """Generate a local embedding using Ollama."""
    payload = json.dumps({
        "model": "nomic-embed-text:latest",
        "input": text,
    }).encode("utf-8")

    request = urllib.request.Request(
        "http://localhost:11434/api/embed",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    with urllib.request.urlopen(request, timeout=10) as response:
        result = json.loads(response.read().decode("utf-8"))

    return result["embeddings"][0]


for document in documents:
    document["embedding"] = get_embedding(document["text"])

print("Documents embedded:", len(documents))
print("Embedding dimensions:", len(documents[0]["embedding"]))

Documents embedded: 3
Embedding dimensions: 768


In [8]:
import math

def cosine_similarity(vector_a, vector_b):
    """Calculate cosine similarity between two vectors."""
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    norm_a = math.sqrt(sum(a * a for a in vector_a))
    norm_b = math.sqrt(sum(b * b for b in vector_b))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)


def search_documents(question, top_k=2):
    """Return the most relevant documents for a question."""
    question_embedding = get_embedding(question)

    ranked_documents = sorted(
        documents,
        key=lambda document: cosine_similarity(
            question_embedding, document["embedding"]
        ),
        reverse=True,
    )

    return [
        {
            "title": document["title"],
            "score": cosine_similarity(
                question_embedding, document["embedding"]
            ),
            "text": document["text"],
        }
        for document in ranked_documents[:top_k]
    ]


results = search_documents("How do embeddings help find relevant information?")

for result in results:
    print(f'{result["title"]} | similarity: {result["score"]:.3f}')
    print(result["text"])
    print()

Embeddings | similarity: 0.718
Embeddings represent text as numerical vectors. Similarity between vectors can help identify relevant document chunks.

RAG Basics | similarity: 0.652
Retrieval-Augmented Generation retrieves relevant information from documents and uses it to help generate grounded answers.



In [11]:
# Verify retrieval returns relevant results without LLM generation
results = search_documents(
    "How do embeddings find relevant information?",
    top_k=2,
)

assert len(results) == 2
assert results[0]["title"] == "Embeddings"
assert all("text" in item and "score" in item for item in results)

print("Retrieval test passed.")
print("Top result:", results[0]["title"])
print("Results returned:", len(results))

Retrieval test passed.
Top result: Embeddings
Results returned: 2


## Results and Limitations

### Working Features
- Ollama API connectivity verified with HTTP 200.
- Local embeddings generated using `nomic-embed-text:latest`.
- Three sample documents embedded with 768-dimensional vectors.
- Cosine similarity used to rank relevant documents.
- Retrieval test passed with the expected top result.

### Model Generation
- Generation requests to `qwen2.5:3b` and `llama3.2:3b` timed out.
- Successful answer generation has not been verified.
- The assistant currently demonstrates document retrieval, not a fully verified end-to-end answer-generation workflow.

### Conclusion
The local embedding and retrieval components work as expected on the sample knowledge base. Further investigation is needed to resolve local LLM generation timeouts.ss

In [12]:
# Test a question unrelated to the sample knowledge base
results = search_documents(
    "What is the capital of Japan?",
    top_k=2,
)

assert len(results) == 2
assert all("title" in item and "score" in item for item in results)

print("Unrelated-question retrieval test passed.")
print("Returned documents:", [item["title"] for item in results])

Unrelated-question retrieval test passed.
Returned documents: ['RAG Basics', 'Embeddings']


In [13]:
def search_documents(question, top_k=2, min_score=0.70):
    """Return relevant documents above a cosine-similarity threshold."""
    question_embedding = get_embedding(question)

    ranked_documents = sorted(
        documents,
        key=lambda document: cosine_similarity(
            question_embedding, document["embedding"]
        ),
        reverse=True,
    )

    results = []
    for document in ranked_documents:
        score = cosine_similarity(
            question_embedding, document["embedding"]
        )
        if score >= min_score:
            results.append({
                "title": document["title"],
                "score": score,
                "text": document["text"],
            })

    return results[:top_k]


relevant_results = search_documents(
    "How do embeddings find relevant information?"
)
unrelated_results = search_documents(
    "What is the capital of Japan?"
)

print("Relevant results:", len(relevant_results))
print("Unrelated results:", len(unrelated_results))

Relevant results: 1
Unrelated results: 0


In [14]:
# Verify relevant and unrelated questions
relevant = search_documents(
    "How do embeddings find relevant information?"
)
unrelated = search_documents(
    "What is the capital of Japan?"
)

assert len(relevant) >= 1
assert relevant[0]["title"] == "Embeddings"
assert len(unrelated) == 0

# Verify the relevance threshold is respected
assert all(item["score"] >= 0.70 for item in relevant)

print("All 4 retrieval checks passed.")

All 4 retrieval checks passed.


## Code Review and Final Validation

### Verified
- Ollama API connectivity: HTTP 200.
- Local embedding generation: 768 dimensions.
- Three sample documents embedded successfully.
- Cosine-similarity retrieval implemented.
- Relevance threshold applied.
- All four retrieval checks passed.
- Relevant query returned the expected Embeddings document.
- Unrelated query returned no results with the current threshold.

### Limitations
- Answer generation using qwen2.5:3b and llama3.2:3b timed out.
- End-to-end LLM answer generation remains unverified.
- The knowledge base currently contains three sample documents.
- The similarity threshold is a heuristic and needs validation on a larger dataset.

### Conclusion
The local embedding and retrieval components work on the sample knowledge base. The next improvement is to resolve local model-generation timeouts and test with real research documents.